# Import Libraries

In [ ]:
!pip install torchsummaryX wandb --quiet

import torch
import torch.nn as nn
import numpy as np
from torchsummaryX import summary
import sklearn
import sklearn.metrics
import gc
import pandas as pd
import os
import wandb
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.8 MB/s eta 0:00:00
Device:  cpu


In [ ]:
### For colab, you can import google drive to save model checkpoints in a folder
# from google.colab import drive
# drive.mount('/content/drive')

# Acquire the Dataset

In [ ]:
# kaggle setups

!pip install --upgrade kaggle
!mkdir /root/.kaggle

with open("/root/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"skyprotector","key":"9783f55e9344bbb8e4f86a3d29936e69"}')
    # Put your kaggle username & key here

!chmod 600 /root/.kaggle/kaggle.json

# Then download the dataset as instructed
!kaggle competitions download -c titanic
!unzip titanic.zip

  0% 0.00/34.1k [00:00<?, ?B/s]
100% 34.1k/34.1k [00:00<00:00, 38.2MB/s]
Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


# Examine the Dataset

In [ ]:
train_data = pd.read_csv("/content/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of women who survived: 0.7420382165605095
% of men who survived: 0.18890814558058924


# Train Val Split (Not Needed for Many Datasets)

In [ ]:
train_data = pd.read_csv("/content/train.csv")
num_items = len(train_data)
split_ind = num_items // 5 * 4

val_data = train_data.iloc[split_ind:num_items]
train_data = train_data.iloc[0:split_ind]

train_data.to_csv("/content/train.csv")
val_data.to_csv("/content/val.csv")

# Dataset Class

In [ ]:
class TitanicDataset(torch.utils.data.Dataset):

    def __init__(self, data_path, partition="train"):

        self.data_path = data_path
        self.partition = partition

        # Normally datasets are not as convienient as a single csv, in which case we usually build a for-loop and iterate over files;
        raw_data = pd.read_csv(self.data_path + "/" + self.partition + ".csv")

        # Preprocess the dataset here using whatever methods; here we only replace NaN with -1 for all columns
        self.full_data = raw_data.fillna(-1)
        sex_map = {"male": 0, "female": 1}
        self.full_data = self.full_data.replace({"Sex": sex_map})

        self.length = len(self.full_data)

    def __len__(self):

        return self.length

    def __getitem__(self, ind):

        entry = self.full_data.iloc[ind]

        survived = entry.loc["Survived"]
        pclass = entry.loc["Pclass"]
        # name = entry.loc["Name"]
        sex = entry.loc["Sex"]
        age = entry.loc["Age"]
        sibsp = entry.loc["SibSp"]
        parch = entry.loc["Parch"]
        # ticket = entry.loc["Ticket"]
        fare = entry.loc["Fare"]
        # cabin = entry.loc["Cabin"]
        # embarked = entry.loc["Embarked"]
        inputs = torch.FloatTensor([pclass, sex, age, sibsp, parch, fare])
        survived = torch.FloatTensor([survived])

        return survived, inputs

In [ ]:
# Basically the same as above, but for test dataset we don't have the target values

class TitanicTestDataset(torch.utils.data.Dataset):

    def __init__(self, data_path):

        self.data_path = data_path

        # Normally datasets are not as convienient as a single csv, in which case we usually build a for-loop and iterate over files;
        raw_data = pd.read_csv(self.data_path + "/test.csv")

        # Preprocess the dataset here using whatever methods; here we only replace NaN with -1 for all columns
        self.full_data = raw_data.fillna(-1)
        sex_map = {"male": 0, "female": 1}
        self.full_data = self.full_data.replace({"Sex": sex_map})

        self.length = len(self.full_data)

    def __len__(self):

        return self.length

    def __getitem__(self, ind):

        entry = self.full_data.iloc[ind]

        pclass = entry.loc["Pclass"]
        # name = entry.loc["Name"]
        sex = entry.loc["Sex"]
        age = entry.loc["Age"]
        sibsp = entry.loc["SibSp"]
        parch = entry.loc["Parch"]
        # ticket = entry.loc["Ticket"]
        fare = entry.loc["Fare"]
        # cabin = entry.loc["Cabin"]
        # embarked = entry.loc["Embarked"]

        inputs = torch.FloatTensor([pclass, sex, age, sibsp, parch, fare])

        return inputs

# Macros

In [ ]:
config = {
    'epochs': 10,
    'batch_size' : 16,
    'learning_rate' : 0.001,
    'architecture' : 'tmp',
    'dropout' : 0.1,
    'step_size' : 3,
    'gamma' : 0.05,
    # Add more as you need them - e.g dropout values, weight decay, scheduler parameters
}

# Dataloaders & Load the Dataset

In [ ]:
content_path = '/content'

train_data = TitanicDataset(content_path, "train")
val_data = TitanicDataset(content_path, "val")
test_data = TitanicTestDataset(content_path)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= True)
val_loader = torch.utils.data.DataLoader(val_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= False)
test_loader = torch.utils.data.DataLoader(test_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= False)

print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Train dataset samples = 712, batches = 45
Val dataset samples = 179, batches = 12
Test dataset samples = 418, batches = 27


In [ ]:
# Testing code to check if your data loaders are working

for i, data in enumerate(train_loader):
    targets, inputs = data
    print(targets.shape, inputs.shape)
    break

torch.Size([16, 1]) torch.Size([16, 6])


# Network Architecture

In [ ]:
class Network(nn.Module):

    def __init__(self, input_size, dropout):

        super(Network, self).__init__()

        output_size = 1 # Why?

        self.model = nn.Sequential(

            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, output_size),

        )

        self.sigmoid = nn.Sigmoid() # Why?

    def forward(self, x):
        out = self.model(x)
        out = self.sigmoid(out)

        return out

# Init Model

In [ ]:
targets, inputs = next(iter(train_loader))

# Model init here
model = Network(inputs.shape[1], config['dropout']).to(device)

# Check your model architecture and parameters
summary(model, inputs.to(device))

                  Kernel Shape Output Shape Params Mult-Adds
Layer                                                       
0_model.Linear_0      [6, 128]    [16, 128]  896.0     768.0
1_model.ReLU_1               -    [16, 128]      -         -
2_model.Dropout_2            -    [16, 128]      -         -
3_model.Linear_3      [128, 1]      [16, 1]  129.0     128.0
4_sigmoid                    -      [16, 1]      -         -
--------------------------------------------------------------
                      Totals
Total params          1.025k
Trainable params      1.025k
Non-trainable params     0.0
Mult-Adds              896.0


/usr/local/lib/python3.10/dist-packages/torchsummaryX/torchsummaryX.py:101: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  df_sum = df.sum()


,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_model.Linear_0,"[6, 128]","[16, 128]",896.0,768.0
1_model.ReLU_1,-,"[16, 128]",NaN,NaN
2_model.Dropout_2,-,"[16, 128]",NaN,NaN
3_model.Linear_3,"[128, 1]","[16, 1]",129.0,128.0
4_sigmoid,-,"[16, 1]",NaN,NaN


# Set up Loss Function, Optimizer, Scheduler

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, config['step_size'], config['gamma'])

# Empty CUDA Cache & Reclaim Memory (Useful When Encountering CUDA OOM Error)

In [ ]:
torch.cuda.empty_cache()
gc.collect()

70

# Train & Eval Functions

In [ ]:
def train(model, optimizer, criterion, dataloader, scaler):

    model.train()
    train_loss = 0.0

    for iter, (targets, inputs) in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        targets = targets.to(device)
        inputs = inputs.to(device)

        # Forward Propagation
        outputs = model(inputs)

        # Loss Calculation
        loss = criterion(outputs, targets)
        train_loss += loss.item()

        # Initialize Gradients
        optimizer.zero_grad()

        # Backward Propagation
        scaler.scale(loss).backward()

        # Gradient Descent
        scaler.step(optimizer)

        scaler.update()

    train_loss /= len(dataloader)
    return train_loss

In [ ]:
def eval(model, dataloader):

    model.eval() # Set model in evaluation mode

    ground_truth_list = []
    outputs_list = []

    for i, (targets, inputs) in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        targets = targets.to(device)
        inputs = inputs.to(device)

        with torch.inference_mode(): # Make sure that there are no gradients computed as we are not training the model now
            # Forward Propagation
            outputs = model(inputs)

        # Store GT & Outputs
        ground_truth_list.extend(targets.cpu().tolist())
        outputs_list.extend(outputs.round().cpu().tolist())

    # Calculate Accuracy
    accuracy = sklearn.metrics.accuracy_score(outputs_list, ground_truth_list)
    return accuracy*100

# Set Up W&B

In [ ]:
wandb.login(key="5da5d90f8b775bd6ff70ae7fbd6d006118133cd6") # API Key is in your wandb account, under settings (wandb.ai/settings)

# Create your wandb run
run = wandb.init(
    name = "simple network",
    reinit = True,
    project = "bootcamp", # Project should be created in your wandb account
    config = config
)

# Save your model architecture as a string with str(model)
model_arch = str(model)

# Save it in a txt file
arch_file = open("model_arch.txt", "w")
file_write = arch_file.write(model_arch)
arch_file.close()

# log it in your wandb run with wandb.save()
wandb.save('model_arch.txt')

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pxc15721489907. Use `wandb login --relogin` to force relogin


['/content/wandb/run-20231220_183810-l67dyark/files/model_arch.txt']

# Experiment

In [ ]:
best_acc = 0.0

scaler = torch.cuda.amp.GradScaler()

for epoch in range(config['epochs']):
    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    train_loss = train(model, optimizer, criterion, train_loader, scaler)
    accuracy = eval(model, val_loader)

    print("\tTrain Loss: {:.4f}".format(train_loss))
    print("\tValidation Accuracy: {:.2f}%".format(accuracy))

    # Log metrics at each epoch in your run
    # wandb.log({"train loss": train_loss, "validation accuracy": accuracy})

    # Save checkpoint if accuracy is better than your current best
    if accuracy >= best_acc:

      # Save checkpoint with information you want
      torch.save({'epoch': epoch,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'loss': train_loss,
              'acc': accuracy},
        './model_checkpoint.pth')

      # Save checkpoint in wandb
      # wandb.save('checkpoint.pth')

    scheduler.step()

    # Optional: Mixed Precision Training with T4/V100/A100/etc. - https://pytorch.org/docs/stable/notes/amp_examples.html

# Finish your wandb run
# run.finish()


Epoch 1/10


/usr/local/lib/python3.10/dist-packages/torch/cuda/amp/grad_scaler.py:125: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


	Train Loss: 1.0969
	Validation Accuracy: 70.95%

Epoch 2/10
	Train Loss: 0.8450
	Validation Accuracy: 72.07%

Epoch 3/10
	Train Loss: 0.9754
	Validation Accuracy: 75.98%

Epoch 4/10
	Train Loss: 0.7719
	Validation Accuracy: 74.30%

Epoch 5/10
	Train Loss: 0.9654
	Validation Accuracy: 74.86%

Epoch 6/10
	Train Loss: 0.7629
	Validation Accuracy: 75.42%

Epoch 7/10
	Train Loss: 0.7959
	Validation Accuracy: 75.98%

Epoch 8/10
	Train Loss: 1.0746
	Validation Accuracy: 76.54%

Epoch 9/10
	Train Loss: 0.7592
	Validation Accuracy: 76.54%

Epoch 10/10
	Train Loss: 0.9701
	Validation Accuracy: 76.54%


# Testing

In [ ]:
def test(model, dataloader):

    model.eval()

    outputs_list = []

    for i, inputs in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        inputs = inputs.to(device)

        with torch.inference_mode(): # Make sure that there are no gradients computed as we are not training the model now
            # Forward Propagation
            outputs = model(inputs)

        # Store Outputs
        outputs_list.extend(outputs.round().type(torch.int64).cpu().tolist())

    return sum(outputs_list, [])

In [ ]:
predictions = test(model, test_loader)

# Create CSV file with predictions
with open(content_path + "/submission.csv", "w+") as f:
    f.write("PassengerId,Survived\n")
    for i in range(len(predictions)):
        f.write("{},{}\n".format(i+892, predictions[i]))

# Submission to Kaggle (if applicable)

In [ ]:
!kaggle competitions submit -c titanic -f submission.csv -m "Message"

100% 2.77k/2.77k [00:00<00:00, 4.31kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster